In [1]:
import os
import json
import pandas as pd
import traceback

In [2]:
from langchain_groq import ChatGroq

In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [4]:
KEY = os.getenv("GROQ_API_KEY")

In [5]:
llm = ChatGroq(groq_api_key=KEY, model_name="openai/gpt-oss-20b", temperature=0.5)

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
import PyPDF2

In [8]:
import re

def clean_json_output(text: str) -> str:
    # Removes ```json ... ``` or plain ``` ... ``` wrappers
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

In [9]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    }
}

In [10]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [11]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [12]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser() |  RunnableLambda(clean_json_output)

In [13]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""


In [14]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [15]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [16]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [17]:
file_path = r"C:\Users\Ojas Pal\mcqgen\data.txt"

In [18]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [19]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [20]:
NUMBER = 5
SUBJECT = "Artificial intelligence"
TONE = "Simple"

In [21]:
callback = UsageMetadataCallbackHandler()

response = generate_evaluate_chain.invoke(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    },
    config={"callbacks": [callback]}
)

print(callback.usage_metadata)

{'openai/gpt-oss-20b': {'total_tokens': 2709, 'input_tokens': 1309, 'output_tokens': 1400, 'output_token_details': {'reasoning': 523}}}


In [25]:
response.get("quiz")

'{\n  "1": {\n    "mcq": "Artificial intelligence is the capability of computational systems to perform tasks typically associated with ______.",\n    "options": {\n      "a": "human emotions",\n      "b": "human intelligence",\n      "c": "human creativity",\n      "d": "human memory"\n    },\n    "correct": "b"\n  },\n  "2": {\n    "mcq": "Which of the following is not listed as a high‑profile AI application in the text?",\n    "options": {\n      "a": "autonomous vehicles",\n      "b": "content generation",\n      "c": "medical diagnosis",\n      "d": "chatbots"\n    },\n    "correct": "c"\n  },\n  "3": {\n    "mcq": "AI researchers use techniques including state space search, formal logic, neural networks, and statistics. Which of the following is NOT mentioned as a technique?",\n    "options": {\n      "a": "genetic algorithms",\n      "b": "operations research",\n      "c": "economics",\n      "d": "formal logic"\n    },\n    "correct": "a"\n  },\n  "4": {\n    "mcq": "When was a

In [27]:
quiz =json.loads(quiz)

In [28]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})


In [29]:
quiz_table_data

[{'MCQ': 'Artificial intelligence is the capability of computational systems to perform tasks typically associated with ______.',
  'Choices': 'a: human emotions | b: human intelligence | c: human creativity | d: human memory',
  'Correct': 'b'},
 {'MCQ': 'Which of the following is not listed as a high‑profile AI application in the text?',
  'Choices': 'a: autonomous vehicles | b: content generation | c: medical diagnosis | d: chatbots',
  'Correct': 'c'},
 {'MCQ': 'AI researchers use techniques including state space search, formal logic, neural networks, and statistics. Which of the following is NOT mentioned as a technique?',
  'Choices': 'a: genetic algorithms | b: operations research | c: economics | d: formal logic',
  'Correct': 'a'},
 {'MCQ': 'When was artificial intelligence founded as an academic discipline?',
  'Choices': 'a: 1945 | b: 1956 | c: 1965 | d: 1975',
  'Correct': 'b'},
 {'MCQ': 'What technology started being used after 2012 that helped accelerate AI?',
  'Choices'

In [31]:
quiz=pd.DataFrame(quiz_table_data)

In [34]:
quiz.to_csv("Artificial intelligence.csv", index=False)

In [1]:
from datetime import datetime
datetime.now().strftime('%m_%d_%Y_%H_%M_%S')

'08_31_2026_09_30_05'